# 📊 Figures pour la Soutenance MSPR

Ce notebook génère les figures à utiliser dans la présentation.

---

In [ ]:
# Imports et configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Style professionnel
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

# Couleurs
COLORS = {
    'day': '#3498db',      # Bleu
    'night': '#9b59b6',     # Violet
    'primary': '#2c3e50',   # Gris foncé
    'accent': '#e74c3c'     # Rouge
}

# Chemin de sortie
OUTPUT_DIR = '/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/'

# Chargement des données
df = pd.read_csv('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/data/processed/OBRAIL_COMPLETE_FINAL.csv', low_memory=False)
print(f"✅ Données chargées: {len(df):,} trains")

---
## 1. Sources de données (Figure pour le contexte)

In [ ]:
# Figure 1: Sources de données
fig, ax = plt.subplots(figsize=(14, 6))

source_counts = df['data_source'].value_counts()

# Noms plus lisibles
source_names = {
    'germany_gtfs': 'Allemagne (DB)',
    'sncf_transilien': 'France (Transilien)',
    'renfe': 'Espagne (Renfe)',
    'sncf_intercités': 'France (Intercités)',
    'trenitalia': 'Italie (Trenitalia)',
    'back_on_track': 'Europe (Night Trains)'
}

labels = [source_names.get(s, s) for s in source_counts.index]
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(source_counts)))

bars = ax.barh(labels, source_counts.values, color=colors, edgecolor='white', linewidth=2)

# Valeurs sur les barres
for bar, val in zip(bars, source_counts.values):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Nombre de trains', fontsize=14)
ax.set_title('Volume de données par source', fontsize=18, fontweight='bold', pad=20)
ax.set_xlim(0, max(source_counts.values) * 1.15)

# Annotation pour Back-on-Track
bot_idx = list(source_counts.index).index('back_on_track')
ax.annotate('Source spécialisée\ntrains de nuit', 
            xy=(source_counts['back_on_track'], bot_idx),
            xytext=(source_counts['back_on_track'] + 15000, bot_idx + 1.5),
            fontsize=11, ha='center',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff3cd', edgecolor='#ffc107'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig1_sources_donnees.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 1 sauvegardée: fig1_sources_donnees.png")

---
## 2. Classification Jour/Nuit (Figure principale)

In [ ]:
# Application de la nouvelle classification
def calc_night_pct(dep, arr):
    try:
        dep_parts = str(dep).split(':')
        arr_parts = str(arr).split(':')
        dep_min = int(dep_parts[0]) * 60 + int(dep_parts[1])
        arr_min = int(arr_parts[0]) * 60 + int(arr_parts[1])
        
        if arr_min <= dep_min:
            duree = (24*60 - dep_min) + arr_min
        else:
            duree = arr_min - dep_min
        
        if duree == 0:
            return 0
        
        nuit_mins = 0
        cur = dep_min
        for _ in range(int(duree)):
            if cur >= 21*60 or cur < 5*60:
                nuit_mins += 1
            cur = (cur + 1) % (24*60)
        
        return (nuit_mins / duree) * 100
    except:
        return 0

def classify_nuit(row):
    if row['has_couchette'] or row['has_sleeper']:
        return 'night'
    if row['duration_hours'] >= 4 and calc_night_pct(row['departure_time'], row['arrival_time']) >= 50:
        return 'night'
    return 'day'

df['classification'] = df.apply(classify_nuit, axis=1)

# Figure 2: Répartition Jour/Nuit
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
ax1 = axes[0]
counts = df['classification'].value_counts()
colors_pie = [COLORS['day'], COLORS['night']]

wedges, texts, autotexts = ax1.pie(
    counts.values, 
    labels=['Jour', 'Nuit'],
    colors=colors_pie,
    autopct=lambda p: f'{p:.1f}%\n({int(p*len(df)/100):,})',
    startangle=90,
    explode=(0, 0.05),
    shadow=True,
    textprops={'fontsize': 14, 'fontweight': 'bold'}
)
autotexts[0].set_color('white')
autotexts[1].set_color('white')
ax1.set_title('Répartition des trains', fontsize=16, fontweight='bold')

# Bar chart par source
ax2 = axes[1]
source_day_night = df.groupby(['data_source', 'classification']).size().unstack(fill_value=0)
source_day_night = source_day_night.reindex(columns=['day', 'night'])

x = np.arange(len(source_day_night))
width = 0.35

bars1 = ax2.bar(x - width/2, source_day_night['day'], width, label='Jour', color=COLORS['day'], edgecolor='white')
bars2 = ax2.bar(x + width/2, source_day_night['night'], width, label='Nuit', color=COLORS['night'], edgecolor='white')

ax2.set_ylabel('Nombre de trains')
ax2.set_title('Trains de jour et nuit par source', fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([source_names.get(s, s) for s in source_day_night.index], rotation=45, ha='right')
ax2.legend(fontsize=12)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig2_repartition_jour_nuit.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 2 sauvegardée: fig2_repartition_jour_nuit.png")

---
## 3. Règles de classification (Figure méthodologie)

In [ ]:
# Figure 3: Visualisation des règles de classification
fig, ax = plt.subplots(figsize=(14, 8))

# Calcul des raisons
def get_raison(row):
    if row['has_couchette'] or row['has_sleeper']:
        return 'Couchettes'
    if row['duration_hours'] >= 4 and calc_night_pct(row['departure_time'], row['arrival_time']) >= 50:
        return 'Durée ≥ 4h + % nuit ≥ 50%'
    return 'Jour (défaut)'

df['raison'] = df.apply(get_raison, axis=1)
raison_counts = df['raison'].value_counts()

# Création du diagramme
colors_raison = {'Jour (défaut)': COLORS['day'], 
                'Durée ≥ 4h + % nuit ≥ 50%': COLORS['night'],
                'Couchettes': '#27ae60'}

bars = ax.barh(raison_counts.index, raison_counts.values, 
               color=[colors_raison[r] for r in raison_counts.index],
               edgecolor='white', linewidth=2)

for bar, val in zip(bars, raison_counts.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', va='center', fontsize=14, fontweight='bold')

ax.set_xlabel('Nombre de trains', fontsize=14)
ax.set_title('Règles de classification', fontsize=18, fontweight='bold', pad=20)

# Légende explicative
ax.text(0.98, 0.02, 
        '🌙 NUIT: Couchettes OU (Durée ≥ 4h ET % nuit ≥ 50%)\n'
        '☀️ JOUR: Tous les autres trains',
        transform=ax.transAxes, fontsize=12, verticalalignment='bottom',
        horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#dee2e6'))

ax.set_xlim(0, max(raison_counts.values) * 1.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig3_regles_classification.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 3 sauvegardée: fig3_regles_classification.png")

---
## 4. Modèle Machine Learning - Matrice de confusion

In [ ]:
# Préparation des données pour le ML
def extract_hour(t):
    try:
        return int(str(t).split(':')[0])
    except:
        return None

df['heure_dep'] = df['departure_time'].apply(extract_hour)
df['heure_arr'] = df['arrival_time'].apply(extract_hour)
df['night_pct'] = df.apply(lambda r: calc_night_pct(r['departure_time'], r['arrival_time']), axis=1)

# Features
feature_cols = ['heure_dep', 'heure_arr', 'duration_hours', 'distance_km', 
                'night_pct', 'has_couchette', 'has_sleeper']

df_ml = df[feature_cols + ['classification']].dropna()
df_ml['target'] = (df_ml['classification'] == 'night').astype(int)

X = df_ml[feature_cols]
y = df_ml['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entraînement Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Figure 4: Matrice de confusion
fig, ax = plt.subplots(figsize=(10, 8))

cm = confusion_matrix(y_test, y_pred)

# Normalisation
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Affichage
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Jour', 'Nuit'], yticklabels=['Jour', 'Nuit'],
            annot_kws={'size': 16, 'weight': 'bold'})

ax.set_xlabel('Prédit', fontsize=14, fontweight='bold')
ax.set_ylabel('Réel', fontsize=14, fontweight='bold')
ax.set_title('Matrice de Confusion - Random Forest', fontsize=18, fontweight='bold', pad=20)

# Métriques
from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

ax.text(0.5, -0.15, f'Accuracy: {acc:.1%}  |  F1-Score: {f1:.1%}',
        transform=ax.transAxes, fontsize=14, ha='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#d4edda', edgecolor='#28a745'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig4_matrice_confusion.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✅ Figure 4 sauvegardée: fig4_matrice_confusion.png")
print(f"   Accuracy: {acc:.1%}, F1: {f1:.1%}")

---
## 5. Importance des features

In [ ]:
# Figure 5: Importance des features
fig, ax = plt.subplots(figsize=(12, 7))

feature_names_display = {
    'heure_dep': 'Heure de départ',
    'heure_arr': 'Heure d\'arrivée',
    'duration_hours': 'Durée (heures)',
    'distance_km': 'Distance (km)',
    'night_pct': '% de nuit',
    'has_couchette': 'Couchettes',
    'has_sleeper': 'Sleepers'
}

importance = pd.DataFrame({
    'feature': [feature_names_display.get(f, f) for f in feature_cols],
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

colors_imp = plt.cm.viridis(importance['importance'] / importance['importance'].max())

bars = ax.barh(importance['feature'], importance['importance'], color=colors_imp, edgecolor='white', linewidth=2)

for bar, val in zip(bars, importance['importance']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{val:.1%}', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Importance', fontsize=14)
ax.set_title('Importance des variables dans le modèle', fontsize=18, fontweight='bold', pad=20)
ax.set_xlim(0, max(importance['importance']) * 1.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig5_importance_features.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 5 sauvegardée: fig5_importance_features.png")

---
## 6. Validation Back-on-Track (Preuve de fiabilité)

In [ ]:
# Figure 6: Validation sur Back-on-Track
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bot = df[df['data_source'] == 'back_on_track']

# Gauche: Caractéristiques des trains Back-on-Track
ax1 = axes[0]

metrics = ['duration_hours', 'distance_km', 'night_pct']
metric_names = ['Durée (h)', 'Distance (km)', '% Nuit']
means = [bot[m].mean() for m in metrics]

colors_metric = ['#3498db', '#e74c3c', '#9b59b6']
bars = ax1.bar(metric_names, means, color=colors_metric, edgecolor='white', linewidth=2)

for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            f'{val:.0f}', ha='center', fontsize=14, fontweight='bold')

ax1.set_ylabel('Valeur moyenne')
ax1.set_title('Caractéristiques des trains de nuit\n(Back-on-Track)', fontsize=16, fontweight='bold')
ax1.set_ylim(0, max(means) * 1.3)

# Droite: Taux de détection
ax2 = axes[1]

# Classification métier sur Back-on-Track
bot_nuit = (bot['classification'] == 'night').sum()
bot_jour = len(bot) - bot_nuit

ax2.pie([bot_jour, bot_nuit], 
       labels=['Jour (erreur)', 'Nuit (correct)'],
       colors=[COLORS['accent'], '#27ae60'],
       autopct=lambda p: f'{p:.1f}%',
       startangle=90,
       explode=(0.1, 0),
       shadow=True,
       textprops={'fontsize': 14, 'fontweight': 'bold'})

ax2.set_title('Validation sur Back-on-Track\n(Vérité terrain)', fontsize=16, fontweight='bold')

# Texte explicatif
fig.text(0.5, -0.02, 'Back-on-Track: Base de données spécialisée trains de nuit → 100% classés NUIT',
         ha='center', fontsize=14, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#d4edda', edgecolor='#28a745'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig6_validation_backontrack.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 6 sauvegardée: fig6_validation_backontrack.png")

---
## 7. Statistiques par pays

In [ ]:
# Figure 7: Statistiques par pays
fig, ax = plt.subplots(figsize=(14, 8))

country_names = {
    'FR': 'France', 'DE': 'Allemagne', 'ES': 'Espagne',
    'IT': 'Italie', 'AT': 'Autriche', 'RO': 'Roumanie',
    'BG': 'Bulgarie', 'SE': 'Suède', 'HU': 'Hongrie'
}

# Grouper par pays et type
df_country = df.groupby(['operator_country', 'classification']).size().unstack(fill_value=0)
df_country = df_country.reindex(columns=['day', 'night'], fill_value=0)
df_country = df_country.sort_values('night', ascending=False).head(8)

x = np.arange(len(df_country))
width = 0.35

bars1 = ax.bar(x - width/2, df_country['day'], width, label='Jour', color=COLORS['day'], edgecolor='white')
bars2 = ax.bar(x + width/2, df_country['night'], width, label='Nuit', color=COLORS['night'], edgecolor='white')

ax.set_ylabel('Nombre de trains', fontsize=14)
ax.set_title('Trains de jour et de nuit par pays', fontsize=18, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels([country_names.get(c, c) for c in df_country.index], fontsize=12)
ax.legend(fontsize=12, loc='upper right')

# Annotations sur les barres nuit
for i, val in enumerate(df_country['night']):
    if val > 0:
        ax.text(i + width/2, val + 5, str(val), ha='center', fontsize=11, fontweight='bold', color=COLORS['night'])

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig7_stats_pays.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 7 sauvegardée: fig7_stats_pays.png")

---
## 8. KPIs récapitulatifs

In [ ]:
# Figure 8: KPIs sous forme de cartes
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

kpis = [
    ('Total trains', f'{len(df):,}', '#3498db'),
    ('Sources', f'{df["data_source"].nunique()}', '#e74c3c'),
    ('Trains de nuit', f'{(df["classification"]=="night").sum():,}', '#9b59b6'),
    ('Pays couverts', f'{df["operator_country"].nunique()}', '#27ae60'),
    ('Durée moyenne nuit', f'{bot["duration_hours"].mean():.1f}h', '#f39c12'),
    ('Distance moyenne nuit', f'{bot["distance_km"].mean():.0f} km', '#1abc9c'),
]

for ax, (title, value, color) in zip(axes, kpis):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    # Cadre
    rect = plt.Rectangle((0.1, 0.1), 0.8, 0.8, fill=True, 
                         facecolor=color, alpha=0.1, edgecolor=color, linewidth=3)
    ax.add_patch(rect)
    
    # Texte
    ax.text(0.5, 0.6, value, ha='center', va='center', fontsize=32, fontweight='bold', color=color)
    ax.text(0.5, 0.3, title, ha='center', va='center', fontsize=16, color='#2c3e50')

plt.suptitle('Indicateurs Clés du Projet', fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fig8_kpis.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figure 8 sauvegardée: fig8_kpis.png")

---
## Récapitulatif des figures

In [ ]:
import os

print("="*70)
print("FIGURES GÉNÉRÉES POUR LA SOUTENANCE")
print("="*70)

figures = [
    ('fig1_sources_donnees.png', 'Sources de données'),
    ('fig2_repartition_jour_nuit.png', 'Répartition Jour/Nuit'),
    ('fig3_regles_classification.png', 'Règles de classification'),
    ('fig4_matrice_confusion.png', 'Matrice de confusion ML'),
    ('fig5_importance_features.png', 'Importance des features'),
    ('fig6_validation_backontrack.png', 'Validation Back-on-Track'),
    ('fig7_stats_pays.png', 'Statistiques par pays'),
    ('fig8_kpis.png', 'KPIs récapitulatifs'),
]

print(f"\n{'Fichier':<35} {'Description':<30} {'Taille':>10}")
print("-"*75)

for filename, desc in figures:
    filepath = OUTPUT_DIR + filename
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024
        print(f"{filename:<35} {desc:<30} {size:>6.1f} KB")
    else:
        print(f"{filename:<35} {desc:<30} {'NON GÉNÉRÉ':>10}")

print("\n" + "="*70)
print("✅ Toutes les figures sont prêtes pour la présentation !")
print("="*70)